# 09 · Señales técnicas y noticias

**Objetivo:** comprobar si lo que pasa después de una señal técnica de compra o de venta (notebook 05) cambia según las noticias de la empresa en los días anteriores. Se comparan las señales precedidas de noticias positivas, neutrales, negativas o de ninguna, medidas con M4, el mejor modelo de sentimiento (notebooks 07 y 08).

*Nada de lo que se muestra aquí es una recomendación de inversión.*

**Plan:**

| Paso | Qué se hace | Estado |
|---|---|---|
| 1 | Las señales del 05 (primer día de cada racha) y sus noticias previas: cuántas señales caen en cada grupo, sin mirar todavía ninguna rentabilidad | hecho |
| 2 | Rentabilidad a 5 y 10 sesiones menos SPY de cada grupo, y las comparaciones con su intervalo | pendiente |

**Entradas:**
- `data/cache/senales.parquet` (notebook 05): las señales de compra y de venta de cada empresa y sesión
- `data/cache/eventos_noticias.parquet` (notebook 08): los eventos de noticias (empresa, sesión y sentimiento) de cada modelo
- `data/cache/precios_spy.parquet` (notebook 08): las sesiones de la bolsa
- `data/cache/precios_historicos.parquet` (notebook 02): para comprobar la volatilidad de algunas empresas

**Salidas:** ninguna; todo se calcula a partir de las entradas.

In [1]:
import numpy as np
import pandas as pd
import plotly.express as px
from dotenv import load_dotenv

from funciones import CARPETA_CACHE, SEMILLA

load_dotenv();

## 1. ¿Qué señales hay y qué noticias tenían antes?

**El diseño se fijó antes de mirar ninguna rentabilidad:**

- **Solo el primer día de cada racha de señales.** La ruptura de Donchian, por ejemplo, se repite varios días seguidos mientras el precio sigue subiendo; contar cada día sería contar la misma señal varias veces.
- **Solo empresas y fechas con noticias:** las 633 empresas con titulares de FNSPID, de octubre de 2016 a junio de 2020.
- **Las noticias previas son las de las 5 sesiones anteriores a la señal**, sin incluir la sesión de la señal: una noticia de ese día podría ser posterior al cierre en el que salta la señal, y eso sería mirar al futuro. **Como sensibilidad, 3 sesiones.** Las dos ventanas se declararon antes de ver resultados.
- **El grupo de la señal** sale del signo de la media del sentimiento de los eventos de noticias de la ventana (los del notebook 08, con M4): **positivas, neutrales, negativas**, o **sin noticias** si no hay ninguno.

Las señales, tal como las guardó el notebook 05: una fila por empresa y sesión, con la señal del día (`compra`, `venta` o vacía) y los mecanismos que la han disparado.

In [2]:
senales = pd.read_parquet(CARPETA_CACHE / "senales.parquet").sort_values(["ticker", "fecha"]).reset_index(drop=True)
print(f"Filas: {len(senales):,} · empresas: {senales['ticker'].nunique()} · del {senales['fecha'].min():%d %b %Y} al {senales['fecha'].max():%d %b %Y}")
print("Señales:", senales["senal"].value_counts().to_dict())
senales[senales["senal"].notna()].head(3)

Filas: 2,281,091 · empresas: 1000 · del 10 Oct 2016 al 09 Oct 2026
Señales: {'compra': 187837, 'venta': 129549}


,fecha,ticker,regimen,liquido,compra_ema,venta_ema,compra_rsi,venta_rsi,compra_donchian,venta_donchian,senal
61,2017-01-06,A,lateral,True,False,False,False,False,True,False,compra
62,2017-01-09,A,lateral,True,False,False,False,False,True,False,compra
64,2017-01-11,A,lateral,True,False,False,False,False,True,False,compra


### 1.1 El primer día de cada racha

Una racha son sesiones seguidas de una empresa con la misma señal. Se queda el primer día: el de una señal distinta de la de la sesión anterior.

In [3]:
senal_anterior = senales.groupby("ticker")["senal"].shift()
primeras = senales[senales["senal"].notna() & (senales["senal"] != senal_anterior)]
print(f"Días con señal: {senales['senal'].notna().sum():,} · primeros días de racha: {len(primeras):,}"
      f" ({len(primeras) / senales['senal'].notna().sum():.0%})")
primeras["senal"].value_counts()

Días con señal: 317,386 · primeros días de racha: 215,928 (68%)


senal
compra    124555
venta      91373
Name: count, dtype: int64

### 1.2 Las empresas y las fechas con noticias

Los eventos de noticias de M4 (notebook 08). Una señal entra si su empresa tiene noticias en FNSPID y si sus 5 sesiones previas caen dentro del periodo de las noticias.

In [4]:
eventos = pd.read_parquet(CARPETA_CACHE / "eventos_noticias.parquet")
eventos = eventos[eventos["modelo"] == "M4 FinBERT"].reset_index(drop=True)
sesiones = pd.DatetimeIndex(np.sort(pd.read_parquet(CARPETA_CACHE / "precios_spy.parquet", columns=["fecha"])["fecha"].unique()))
numero_de_sesion = pd.Series(np.arange(len(sesiones)), index=sesiones)

VENTANA, VENTANA_SENSIBILIDAD = 5, 3   # sesiones previas a la señal
primera_sesion_con_ventana = sesiones[numero_de_sesion[eventos["sesion"].min()] + VENTANA]
ultima_sesion_con_noticias = eventos["sesion"].max()

estudio = primeras[primeras["ticker"].isin(eventos["ticker"].unique())
                   & primeras["fecha"].between(primera_sesion_con_ventana, ultima_sesion_con_noticias)].copy()
estudio["sesion_numero"] = estudio["fecha"].map(numero_de_sesion)
print(f"Eventos de noticias de M4: {len(eventos):,} de {eventos['ticker'].nunique()} empresas")
print(f"Señales en el estudio: {len(estudio):,} de {estudio['ticker'].nunique()} empresas,"
      f" del {estudio['fecha'].min():%d %b %Y} al {estudio['fecha'].max():%d %b %Y} · {estudio['senal'].value_counts().to_dict()}")

Eventos de noticias de M4: 111,709 de 633 empresas
Señales en el estudio: 54,001 de 621 empresas, del 05 Jan 2017 al 11 Jun 2020 · {'compra': 32325, 'venta': 21676}


### 1.3 ¿Hay que quitar alguna empresa por tener el precio congelado?

En el notebook 05 aparecieron 7 empresas (TECH, MKTX, AES, ACA, CZR, LNTH y PEN) con un rango diario de solo el 0,25–0,5 % del precio, el patrón de una oferta de compra pendiente; con el precio anclado, una señal técnica no significa nada. Pero eso se midió en el último año (2025–2026), y aquí se estudia 2016–2020. Se mira el rango diario mediano (máximo menos mínimo, sobre el cierre) de cada empresa en el periodo del estudio:

In [5]:
precios = pd.read_parquet(CARPETA_CACHE / "precios_historicos.parquet", columns=["fecha", "ticker", "high", "low", "close"])
precios = precios[precios["fecha"].between(estudio["fecha"].min(), estudio["fecha"].max())]
rango_mediano = ((precios["high"] - precios["low"]) / precios["close"]).groupby(precios["ticker"]).median()

SIETE_DEL_05 = ["TECH", "MKTX", "AES", "ACA", "CZR", "LNTH", "PEN"]
print(f"Rango diario mediano de todas las empresas: {rango_mediano.median():.2%}")
print("Las 7 del notebook 05:", {ticker: f"{rango_mediano[ticker]:.2%}" for ticker in SIETE_DEL_05})
casi_quietas = rango_mediano[rango_mediano < 0.006]
print(f"Empresas con un rango por debajo del 0,6 %: {sorted(casi_quietas.index)}"
      f" · con señales en el estudio: {sorted(set(casi_quietas.index) & set(estudio['ticker']))}")

Rango diario mediano de todas las empresas: 1.90%
Las 7 del notebook 05: {'TECH': '1.96%', 'MKTX': '2.22%', 'AES': '1.85%', 'ACA': '3.18%', 'CZR': '3.30%', 'LNTH': '3.98%', 'PEN': '2.94%'}
Empresas con un rango por debajo del 0,6 %: ['ALM', 'AMCR', 'ASTS', 'FER', 'FERG', 'FLUT', 'HIMS', 'SW', 'VRT'] · con señales en el estudio: []


### 1.4 Las noticias de las sesiones previas a cada señal

Para cada señal se buscan los eventos de noticias de su empresa en las sesiones anteriores (5, y 3 como sensibilidad), sin la de la señal. La media de su sentimiento decide el grupo.

In [6]:
NOMBRE_DEL_GRUPO = {1: "positivas", 0: "neutrales", -1: "negativas"}


def grupo_por_noticias_previas(senales_estudio: pd.DataFrame, ventana: int) -> pd.Series:
    """Grupo de cada señal según la media del sentimiento de los eventos de sus `ventana` sesiones anteriores."""
    pares = senales_estudio[["ticker", "fecha", "sesion_numero"]].merge(
        eventos.assign(sesion_evento=eventos["sesion"].map(numero_de_sesion))[["ticker", "sesion_evento", "sentimiento_medio"]],
        on="ticker")
    en_ventana = pares[(pares["sesion_evento"] >= pares["sesion_numero"] - ventana) & (pares["sesion_evento"] < pares["sesion_numero"])]
    media = en_ventana.groupby(["ticker", "fecha"])["sentimiento_medio"].mean()
    grupo = np.sign(media).astype(int).map(NOMBRE_DEL_GRUPO)
    return (senales_estudio.set_index(["ticker", "fecha"]).index.map(grupo).to_series(index=senales_estudio.index)
            .fillna("sin noticias"))


estudio["grupo"] = grupo_por_noticias_previas(estudio, VENTANA)
estudio["grupo_3_sesiones"] = grupo_por_noticias_previas(estudio, VENTANA_SENSIBILIDAD)

ORDEN = ["positivas", "neutrales", "negativas", "sin noticias"]
for ventana, columna in [(VENTANA, "grupo"), (VENTANA_SENSIBILIDAD, "grupo_3_sesiones")]:
    tabla = pd.crosstab(estudio["senal"], estudio[columna])[ORDEN]
    print(f"Noticias en las {ventana} sesiones previas:")
    display(pd.concat([tabla, (100 * tabla.div(tabla.sum(axis=1), axis=0)).round(1).add_suffix(" (%)")], axis=1))

Noticias en las 5 sesiones previas:


grupo,positivas,neutrales,negativas,sin noticias,positivas (%),neutrales (%),negativas (%),sin noticias (%)
senal,,,,,,,,
compra,7925,7977,2343,14080,24.5,24.7,7.2,43.6
venta,4279,5475,2718,9204,19.7,25.3,12.5,42.5


Noticias en las 3 sesiones previas:


grupo_3_sesiones,positivas,neutrales,negativas,sin noticias,positivas (%),neutrales (%),negativas (%),sin noticias (%)
senal,,,,,,,,
compra,5616,6914,1622,18173,17.4,21.4,5.0,56.2
venta,2992,4656,1976,12052,13.8,21.5,9.1,55.6


**Conclusión:** de 215.928 primeros días de racha quedan **54.001 señales** en las 621 empresas con noticias (32.325 compras y 21.676 ventas, de enero de 2017 a junio de 2020). No hace falta quitar ninguna empresa: las 7 del notebook 05 se movían con normalidad en 2016–2020 (rango diario del 1,9 % al 4,0 %), y las 9 que entonces casi no se movían (varias eran sociedades instrumentales a unos 10 dólares) no tienen noticias, así que no entran.

Con la ventana de 5 sesiones, el **57 % de las señales tiene noticias previas**. Los grupos tienen tamaños suficientes para comparar; el más pequeño son las compras con noticias negativas (2.343). Ya se ve una relación: **las compras llegan más a menudo después de noticias positivas** (24,5 %, frente al 7,2 % de negativas), y las ventas tienen proporcionalmente más negativas (12,5 %). Es lógico: las señales técnicas saltan con los movimientos del precio, y esos movimientos coinciden con las noticias. Con 3 sesiones, el reparto es parecido, con menos señales con noticias (44 %).

## Comprobaciones

In [7]:
# Señales: primer día de racha, solo empresas con noticias y fechas con la ventana completa, y grupos sin mirar al futuro
assert estudio["senal"].isin(["compra", "venta"]).all() and not estudio.duplicated(["ticker", "fecha"]).any()
anterior = senales.set_index(["ticker", "fecha"])["senal"].groupby(level="ticker").shift()
assert (anterior.reindex(estudio.set_index(["ticker", "fecha"]).index).to_numpy() != estudio["senal"].to_numpy()).all()
assert estudio["ticker"].isin(eventos["ticker"]).all()
assert (estudio["fecha"] >= primera_sesion_con_ventana).all() and (estudio["fecha"] <= ultima_sesion_con_noticias).all()
assert estudio["grupo"].isin(ORDEN).all() and estudio["grupo_3_sesiones"].isin(ORDEN).all()

# Una señal sin noticias en 5 sesiones tampoco las tiene en 3; y un ejemplo recalculado a mano
assert (estudio.loc[estudio["grupo"] == "sin noticias", "grupo_3_sesiones"] == "sin noticias").all()
ejemplo = estudio[estudio["grupo"] != "sin noticias"].sample(1, random_state=SEMILLA).iloc[0]
del_ejemplo = eventos[(eventos["ticker"] == ejemplo["ticker"])
                      & eventos["sesion"].isin(sesiones[ejemplo["sesion_numero"] - VENTANA : ejemplo["sesion_numero"]])]
assert NOMBRE_DEL_GRUPO[int(np.sign(del_ejemplo["sentimiento_medio"].mean()))] == ejemplo["grupo"]
assert (del_ejemplo["sesion"] < ejemplo["fecha"]).all()   # todas las noticias son anteriores a la señal
print("Señales y noticias previas: OK")

Señales y noticias previas: OK
